In [6]:
import sys
!{sys.executable} -m pip install xgboost


  Using cached xgboost-3.1.2-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.1.2-py3-none-win_amd64.whl (72.0 MB)


In [7]:
import sys
print(sys.executable)

try:
    import xgboost
    print("XGBoost OK:", xgboost.__version__)
except ImportError as e:
    print("ERROR importando xgboost:", e)


c:\Users\CiprianoOss\anaconda3\envs\tp4_ia\python.exe
XGBoost OK: 3.1.2


In [1]:
import sys
print(sys.executable)  

!{sys.executable} -m pip install -U scikit-learn

c:\Users\CiprianoOss\anaconda3\envs\tp4_ia\python.exe


In [2]:
import sys, os

ROOT = os.path.abspath("../")
SRC = os.path.abspath("../src")

sys.path.append(ROOT)
sys.path.append(SRC)

print("Added to path:")
print(ROOT)
print(SRC)

Added to path:
c:\Users\CiprianoOss\Desktop\alquileres-main
c:\Users\CiprianoOss\Desktop\alquileres-main\src


In [3]:
from src.utils import DEV_SET_CLEAN_PATH, TARGET

print(DEV_SET_CLEAN_PATH)
print(TARGET)

data/processed/dev_set_clean.csv
precio_pesos_constantes


In [4]:
import pandas as pd
import numpy as np

from src.utils import DEV_SET_CLEAN_PATH, TARGET  # TARGET = "precio_pesos_constantes"

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

# ============================================================
# 1) Cargar datos limpios
# ============================================================
df = pd.read_csv("../" + DEV_SET_CLEAN_PATH)
print("Shape original:", df.shape)

# ============================================================
# 2) Usar SOLO columnas numéricas (sklearn no banca strings)
# ============================================================
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Columnas numéricas:", numeric_cols)

target_col = TARGET       # "precio_pesos_constantes"
feature_cols = [c for c in numeric_cols if c != target_col]

# Dataset final sin NaN
df_model = df[feature_cols + [target_col]].dropna()
print("Shape después de dropna:", df_model.shape)

X = df_model[feature_cols].values
y = df_model[target_col].values

# ============================================================
# 3) Train / test split (80/20)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================================
# 4) Modelos: Árbol de decisión y Regresión lineal
# ============================================================
tree = DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=20,
    random_state=42
)

linreg = LinearRegression()

# Entrenar modelos
tree.fit(X_train, y_train)
linreg.fit(X_train, y_train)

# ============================================================
# 5) Evaluación
# ============================================================
def evaluar(nombre, modelo):
    y_pred_train = modelo.predict(X_train)
    y_pred_test  = modelo.predict(X_test)

    # RMSE actualizado (sin squared)
    rmse_train = root_mean_squared_error(y_train, y_pred_train)
    rmse_test  = root_mean_squared_error(y_test,  y_pred_test)

    # R²
    r2_train = r2_score(y_train, y_pred_train)
    r2_test  = r2_score(y_test,  y_pred_test)

    # Resultados
    print(f"\n=== {nombre} ===")
    print(f"Train RMSE: {rmse_train:,.2f} | R²: {r2_train:.3f}")
    print(f"Test  RMSE: {rmse_test:,.2f} | R²: {r2_test:.3f}")

# Ejecutar evaluación
evaluar("Árbol de decisión", tree)
evaluar("Regresión lineal",  linreg)


Shape original: (270722, 30)
Columnas numéricas: ['id_grid', 'STotalM2', 'SConstrM2', 'Dormitorios', 'Banos', 'Ambientes', 'Antiguedad', 'Cocheras', 'LONGITUDE', 'LATITUDE', 'precio_pesos_constantes', 'year', 'mes_listing']
Shape después de dropna: (230486, 13)

=== Árbol de decisión ===
Train RMSE: 666,617.74 | R²: 0.071
Test  RMSE: 638,287.73 | R²: 0.060

=== Regresión lineal ===
Train RMSE: 690,384.93 | R²: 0.004
Test  RMSE: 656,880.95 | R²: 0.004


In [8]:
from xgboost import XGBRegressor

# Modelo XGBoost
xgb = XGBRegressor(
    n_estimators=500,          # cantidad de árboles
    learning_rate=0.05,       # tasa de aprendizaje
    max_depth=6,              # profundidad máxima de cada árbol
    subsample=0.8,            # muestreo filas
    colsample_bytree=0.8,     # muestreo columnas
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1                 # usa todos los cores
)

# Entrenar
xgb.fit(X_train, y_train)

evaluar("XGBoost", xgb)



=== XGBoost ===
Train RMSE: 435,553.39 | R²: 0.603
Test  RMSE: 483,528.38 | R²: 0.461


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importances = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 10))
importances.head(20).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Importancia de features - XGBoost")
plt.tight_layout()
plt.show()
